<img src="https://hilpisch.com/tds_logo.png" width="20%" align="right">

# The Data Scientist
## Book 3 · Statistics, Machine Learning, and Model Evaluation
### Notebook 02 · Titanic Baseline
© Dr. Yves J. Hilpisch<br>
AI-Powered by GPT 5.x<br> The Python Quants GmbH | https://tpq.io<br>

https://thedatascientist.dev | https://linktr.ee/dyjh

### Why this notebook exists
The chapter uses the Titanic example to show how a small supervised learning workflow
fits together: load data, define features, fit a model, and inspect the result.

## Setup
We make the notebook portable first so the helper module, the data, and any
later outputs are all resolved from the same book root.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

BOOK_NAME = "3_data"
REPO_URL = "https://github.com/yhilpisch/tdscode"

cwd = Path.cwd().resolve()
BOOK_ROOT = None
for candidate in (cwd, *cwd.parents):
    if candidate.name == BOOK_NAME and (candidate / "notebooks").exists():
        BOOK_ROOT = candidate
        break
    if (candidate / BOOK_NAME / "notebooks").exists():
        BOOK_ROOT = candidate / BOOK_NAME
        break

if BOOK_ROOT is None:
    repo_root = Path("/content/tdscode")
    if not repo_root.exists():
        # Clone the companion repo once in Colab.
        subprocess.run(
            ["git", "clone", "--depth", "1", REPO_URL, str(repo_root)],
            check=True,
        )
    BOOK_ROOT = repo_root / BOOK_NAME

os.chdir(BOOK_ROOT)

# Make the book root and the helper folder importable.
for path in (BOOK_ROOT, BOOK_ROOT / "code"):
    if path.exists() and str(path) not in sys.path:
        sys.path.insert(0, str(path))

requirements = BOOK_ROOT / "requirements.txt"
if requirements.exists() and "google.colab" in sys.modules:
    # Keep Colab aligned with the book dependencies.
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements)],
        check=True,
    )

print("Book root:", BOOK_ROOT)


## Load the Baseline Helper
The helper module keeps the reusable Titanic modeling logic in one place so the
notebook can focus on the workflow rather than file plumbing.


In [ ]:
from pathlib import Path
import importlib.util

import pandas as pd
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

module_path = BOOK_ROOT / 'code' / 'titanic_baseline.py'
spec = importlib.util.spec_from_file_location('titanic_baseline', module_path)
assert spec is not None and spec.loader is not None
titanic_baseline = importlib.util.module_from_spec(spec)
spec.loader.exec_module(titanic_baseline)

print(f'Loaded {module_path.resolve()}')


## Inspect the Weighted Titanic Table
Before fitting a model, we inspect the aggregated table and the weighted target
counts. That makes the training data visible instead of treating it as a black
box.


In [ ]:
df = titanic_baseline.load_titanic_table()

print(df.head().to_string(index=False))
print()
print('weighted target counts:')
print(df.groupby('survived')['freq'].sum())
print()
print('feature counts:')
feature_counts = (
    df.groupby(titanic_baseline.FEATURE_COLUMNS)['freq']
    .sum()
    .sort_values(ascending=False)
    .head(8)
)
print(feature_counts)


## Fit a Weighted Baseline Pipeline
The next cell separates features, target, and weights, then fits the baseline
pipeline on a held-out split and reports the main classification metrics.


In [ ]:
X = df[titanic_baseline.FEATURE_COLUMNS]
y = df[titanic_baseline.TARGET_COLUMN]
w = df[titanic_baseline.WEIGHT_COLUMN]

X_train, X_test, y_train, y_test, w_train, w_test = train_test_split(
    X,
    y,
    w,
    test_size=0.25,
    random_state=42,
    stratify=y,
)

pipe = titanic_baseline.build_pipeline()
pipe.fit(X_train, y_train, model__sample_weight=w_train)
preds = pipe.predict(X_test)

print('accuracy =', round(accuracy_score(y_test, preds, sample_weight=w_test), 3))
print(confusion_matrix(y_test, preds, sample_weight=w_test))
print()
print(classification_report(y_test, preds, sample_weight=w_test, digits=3))


## Inspect the Held-Out Predictions
Metrics summarize the model, but a preview of actual versus predicted rows helps
the delegate see what the model is getting right and wrong.


In [ ]:
preview = X_test.copy()
preview['actual'] = y_test.values
preview['predicted'] = preds

print(preview.to_string(index=False))


### Where We Are Heading Next
Chapter 5 widens the lens to evaluation itself: metrics, confusion matrices, and the
discipline of comparing baselines fairly.

<img src="https://hilpisch.com/tds_logo.png" width="20%" align="right">